# Session 6: Narrative Layer and Full Pipeline

This notebook turns structured explanation sentences into short narrative summaries. It connects the full pipeline from data preparation and forecasting to fuzzy labels, structured explanations, and final human-readable output.

The goal is to produce a clean end-to-end MVP.


In [1]:
import pandas as pd

explanations = pd.read_csv("../src/data/session05_structured_explanations.csv")
explanations

,Model,MAE,RMSE,MAPE,MPE,DA,MAE_Label,MPE_Label,MAPE_Label,DA_Label,Combined_Label,Structured_Explanation
0,Naive,1827.63,2165.77,11.980,5.419,0.00,high error,slight overprediction,high mape,low DA,"high error, slight overprediction",The model has high absolute error with a sligh...
1,Seasonal Naive,1352.16,1719.28,8.944,-0.571,59.77,medium error,neutral,medium mape,medium DA,"medium error, neutral",The model shows moderate absolute error with n...
2,Linear Regression,1238.04,1505.50,7.936,2.908,50.58,low error,slight overprediction,medium mape,low DA,"low error, slight overprediction",The model achieves low absolute error with a s...
3,ETS,1546.33,1812.99,10.018,5.249,66.67,medium error,slight overprediction,medium mape,medium DA,"medium error, slight overprediction",The model shows moderate absolute error with a...
4,HWES (damped),1517.14,1779.22,9.821,5.222,67.82,medium error,slight overprediction,medium mape,medium DA,"medium error, slight overprediction",The model shows moderate absolute error with a...
5,SARIMA,1434.99,1704.84,9.209,4.487,66.67,medium error,slight overprediction,medium mape,medium DA,"medium error, slight overprediction",The model shows moderate absolute error with a...
6,Prophet,1091.53,1320.09,7.053,2.614,65.52,low error,slight overprediction,low mape,medium DA,"low error, slight overprediction",The model achieves low absolute error with a s...


In [2]:
def narrative_summary(model, mae_label, mpe_label, mape_label, da_label, structured_text):
    # Opening sentence based on overall accuracy tier
    if mae_label == "low error" and mape_label == "low mape":
        opening = f"{model} is one of the stronger models in this comparison."
    elif mae_label == "low error":
        opening = f"{model} performs well in terms of absolute error."
    elif mae_label == "medium error":
        opening = f"{model} shows moderate forecasting accuracy."
    else:
        opening = f"{model} is among the weaker models in this evaluation."

    # Directional skill closing — adds the operational implication only;
    # the directional-accuracy fact itself is already stated in structured_text.
    da_closing = {
        "medium DA": "This level of directional accuracy is generally useful for operational scheduling.",
        "low DA":    "This makes it less suited for scheduling decisions that depend on knowing whether demand will rise or fall.",
        "high DA":   "This makes it well suited for scheduling decisions that depend on the direction of demand change.",
    }.get(da_label, "")

    return f"{opening} {structured_text} {da_closing}".strip()

In [3]:
explanations["Narrative"] = explanations.apply(
    lambda r: narrative_summary(
        r["Model"], r["MAE_Label"], r["MPE_Label"], r["MAPE_Label"], r["DA_Label"],
        r["Structured_Explanation"]
    ),
    axis=1
)
explanations[["Model", "Structured_Explanation", "Narrative"]]

,Model,Structured_Explanation,Narrative
0,Naive,The model has high absolute error with a sligh...,Naive is among the weaker models in this evalu...
1,Seasonal Naive,The model shows moderate absolute error with n...,Seasonal Naive shows moderate forecasting accu...
2,Linear Regression,The model achieves low absolute error with a s...,Linear Regression performs well in terms of ab...
3,ETS,The model shows moderate absolute error with a...,ETS shows moderate forecasting accuracy. The m...
4,HWES (damped),The model shows moderate absolute error with a...,HWES (damped) shows moderate forecasting accur...
5,SARIMA,The model shows moderate absolute error with a...,SARIMA shows moderate forecasting accuracy. Th...
6,Prophet,The model achieves low absolute error with a s...,Prophet is one of the stronger models in this ...


In [4]:
for _, row in explanations.iterrows():
    print(row["Narrative"])
    print()


Naive is among the weaker models in this evaluation. The model has high absolute error with a slight tendency to overpredict demand. Its percentage error is high, limiting usefulness for demand-sensitive decisions. It struggles to correctly identify whether demand will rise or fall. This makes it less suited for scheduling decisions that depend on knowing whether demand will rise or fall.

Seasonal Naive shows moderate forecasting accuracy. The model shows moderate absolute error with no consistent directional bias. Its percentage error is moderate, acceptable for operational planning. It correctly identifies the direction of demand change roughly two-thirds of the time. This level of directional accuracy is generally useful for operational scheduling.

Linear Regression performs well in terms of absolute error. The model achieves low absolute error with a slight tendency to overpredict demand. Its percentage error is moderate, acceptable for operational planning. It struggles to corre

In [5]:
explanations.to_csv("../src/data/session06_narratives.csv", index=False)